In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning --quiet

In [ ]:
"""TFT v3 - 하이퍼파라미터 튜닝 (v2 대비 스케일업)
v2: hidden=128, heads=4, lstm=1, dropout=0.2, batch=256, lr=5e-4  -> DA 52.1%
v3: hidden=256, heads=8, lstm=2, dropout=0.15, batch=512, lr=3e-4 + label smoothing
"""
import warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

warnings.filterwarnings("ignore")
pl.seed_everything(42)
print(f"PyTorch: {torch.__version__}, Lightning: {pl.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "tft_v3"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

TRAIN_END, VAL_START, VAL_END, TEST_START = "2024-06-30", "2024-07-01", "2024-12-31", "2025-01-01"
MAX_ENCODER_LENGTH = 60       # v2: 30 -> v3: 60 (장기 패턴)
BATCH_SIZE = 512               # v2: 256 -> v3: 512
MAX_EPOCHS = 80                # v2: 50 -> v3: 80
LEARNING_RATE = 3e-4           # v2: 5e-4 -> v3: 3e-4

# TFT v3 하이퍼파라미터 (스케일업)
HIDDEN_SIZE = 256              # v2: 128 -> v3: 256
ATTENTION_HEADS = 8            # v2: 4 -> v3: 8
NUM_LSTM_LAYERS = 2            # v2: 1 -> v3: 2
DROPOUT = 0.15                 # v2: 0.2 -> v3: 0.15
LABEL_SMOOTHING = 0.05         # 새로 추가
GRADIENT_CLIP_VAL = 0.1

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

print(f"TFT v3 하이퍼파라미터 튜닝")
print(f"  v2 -> v3 변경:")
print(f"    encoder: 30 -> {MAX_ENCODER_LENGTH}")
print(f"    hidden: 128 -> {HIDDEN_SIZE}")
print(f"    heads: 4 -> {ATTENTION_HEADS}")
print(f"    lstm: 1 -> {NUM_LSTM_LAYERS}")
print(f"    batch: 256 -> {BATCH_SIZE}")
print(f"    dropout: 0.2 -> {DROPOUT}")
print(f"    label_smoothing: {LABEL_SMOOTHING}")

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)

n0 = (df["target_5d"]==0).sum(); n1 = (df["target_5d"]==1).sum()
CLASS_WEIGHTS = torch.tensor([len(df)/(2*n0), len(df)/(2*n1)], dtype=torch.float32)
print(f"rows: {len(df):,} | features: {N_FEATURES} | weights: {CLASS_WEIGHTS.tolist()}")

In [ ]:
# ===== 분할 =====
train_df = df[df["date"]<=TRAIN_END]
print(f"학습: {len(train_df):,} | 검증: {len(df[(df['date']>=VAL_START)&(df['date']<=VAL_END)]):,} | 테스트: {len(df[df['date']>=TEST_START]):,}")

In [ ]:
# ===== TFT v3 모델 (v2 구조 + 스케일업 + Label Smoothing) =====

class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Linear(d, d)
        self.gate = nn.Linear(d, d)
    def forward(self, x):
        return self.fc(x) * torch.sigmoid(self.gate(x))

class GatedResidualNetwork(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)
        self.gate = GatedLinearUnit(d_out)
        self.norm = nn.LayerNorm(d_out)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Linear(d_in, d_out) if d_in != d_out else nn.Identity()
    def forward(self, x):
        residual = self.skip(x)
        h = self.dropout(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h) + residual)

class VariableSelectionNetwork(nn.Module):
    def __init__(self, n_vars, d_model, dropout=0.1):
        super().__init__()
        self.grns = nn.ModuleList([GatedResidualNetwork(d_model, d_model, d_model, dropout) for _ in range(n_vars)])
        self.softmax_grn = GatedResidualNetwork(n_vars * d_model, d_model, n_vars, dropout)
    def forward(self, x):
        B, S, V, D = x.shape
        processed = torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)], dim=2)
        flat = x.reshape(B, S, V * D)
        weights = torch.softmax(self.softmax_grn(flat), dim=-1).unsqueeze(-1)
        return (processed * weights).sum(dim=2)

class TFTv3(pl.LightningModule):
    def __init__(self, n_feat, seq_len=60, d_model=256, n_heads=8, n_lstm=2,
                 dropout=0.15, n_cls=2, lr=3e-4, cw=None, label_smoothing=0.05):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"])
        self.lr = lr
        self.n_feat = n_feat

        self.feat_embed = nn.Linear(1, d_model)
        self.vsn = VariableSelectionNetwork(n_feat, d_model, dropout)
        self.lstm = nn.LSTM(d_model, d_model, n_lstm, batch_first=True, dropout=dropout)

        # Multi-head Self-Attention with causal mask
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.attn_gate = GatedLinearUnit(d_model)

        # Post-attention GRN
        self.grn_post = GatedResidualNetwork(d_model, d_model, d_model, dropout)

        # 출력
        self.grn_out = GatedResidualNetwork(d_model, d_model, d_model, dropout)
        self.head = nn.Linear(d_model, n_cls)
        self.loss_fn = nn.CrossEntropyLoss(weight=cw, label_smoothing=label_smoothing) if cw is not None else nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, x):
        B, S, F = x.shape
        x = self.feat_embed(x.unsqueeze(-1))  # (B,S,F,D)
        x = self.vsn(x)  # (B,S,D)
        x, _ = self.lstm(x)  # (B,S,D)

        # Causal self-attention (현재+과거만 참조)
        causal_mask = torch.triu(torch.ones(S, S, device=x.device), diagonal=1).bool()
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = self.attn_norm(x + self.attn_gate(attn_out))
        x = self.grn_post(x)

        # 마지막 타임스텝
        x = self.grn_out(x[:, -1, :])
        return self.head(x)

    def training_step(self, batch, _):
        x,y = batch; loss=self.loss_fn(self(x),y); self.log("train_loss",loss,prog_bar=True); return loss
    def validation_step(self, batch, _):
        x,y = batch; logits=self(x); self.log("val_loss",self.loss_fn(logits,y),prog_bar=True)
        self.log("val_acc",(torch.argmax(logits,1)==y).float().mean(),prog_bar=True)
    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-3)  # 더 강한 정규화
        sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2, eta_min=1e-6)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "interval": "epoch"}}

print("TFT v3 정의 완료 (VSN + 2-layer LSTM + Causal Attention + Label Smoothing)")

In [ ]:
# ===== 데이터셋 =====
def make_samples(full_df, start, end, seq_len, feats):
    samples=[]; s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e: samples.append((v[i-seq_len:i],t[i]))
    return samples

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

tr=SeqDS(make_samples(df,"2008-01-01",TRAIN_END,MAX_ENCODER_LENGTH,CLEANED_FEATURES))
va=SeqDS(make_samples(df,VAL_START,VAL_END,MAX_ENCODER_LENGTH,CLEANED_FEATURES))
te=SeqDS(make_samples(df,TEST_START,None,MAX_ENCODER_LENGTH,CLEANED_FEATURES))
trl=DataLoader(tr,batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
val=DataLoader(va,batch_size=BATCH_SIZE*2,shuffle=False,num_workers=0)
tel=DataLoader(te,batch_size=BATCH_SIZE*2,shuffle=False,num_workers=0)
print(f"학습: {len(tr):,} | 검증: {len(va):,} | 테스트: {len(te):,}")

In [ ]:
# ===== 모델 초기화 =====
model = TFTv3(N_FEATURES, MAX_ENCODER_LENGTH, HIDDEN_SIZE, ATTENTION_HEADS,
              NUM_LSTM_LAYERS, DROPOUT, 2, LEARNING_RATE, CLASS_WEIGHTS, LABEL_SMOOTHING)
print(f"파라미터: {sum(p.numel() for p in model.parameters())/1e3:.1f}K")
print(f"\nv2 (969K) -> v3 ({sum(p.numel() for p in model.parameters())/1e3:.1f}K)")

In [ ]:
# ===== 학습 =====
es = EarlyStopping(monitor="val_loss", patience=10, verbose=True, mode="min")  # patience 8->10
trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1, gradient_clip_val=GRADIENT_CLIP_VAL,
    callbacks=[es, LearningRateMonitor("epoch")], enable_progress_bar=True, log_every_n_steps=50)
print("TFT v3 학습 시작...")
trainer.fit(model, train_dataloaders=trl, val_dataloaders=val)
print(f"완료! Best val_loss: {es.best_score:.4f}")

In [ ]:
# ===== 검증 평가 =====
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
best = TFTv3.load_from_checkpoint(trainer.checkpoint_callback.best_model_path, strict=False)
best.eval(); best.cuda()

def predict_all(m, loader):
    ps,ls=[],[]
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(m(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps),np.array(ls)

vp,va2=predict_all(best,val); vb=(vp>=0.5).astype(int)
da=accuracy_score(va2,vb)
try: auc=roc_auc_score(va2,vp)
except: auc=float("nan")
print("="*60)
print(f"  TFT v3 검증: DA={da*100:.2f}%, AUC={auc:.4f}, 양성={vb.mean()*100:.1f}%")
print("="*60)
print(classification_report(va2,vb,target_names=["하락","상승"]))
print(f"비교: TFT v2 DA=55.1%/52.1%, LSTM=48.9%, LightGBM=50.8%")

In [ ]:
# ===== 테스트 =====
tp,ta2=predict_all(best,tel); tb=(tp>=0.5).astype(int)
tda=accuracy_score(ta2,tb)
try: tauc=roc_auc_score(ta2,tp)
except: tauc=float("nan")
print("="*60)
print(f"  TFT v3 테스트: DA={tda*100:.2f}%, AUC={tauc:.4f}, 양성={tb.mean()*100:.1f}%")
print("="*60)
print(classification_report(ta2,tb,target_names=["하락","상승"]))

In [ ]:
# ===== 저장 =====
import json; from datetime import datetime
trainer.save_checkpoint(str(MODEL_SAVE_PATH/"best.ckpt"))
pd.DataFrame({"prob":tp,"actual":ta2,"pred":tb}).to_parquet(str(MODEL_SAVE_PATH/f"pred_{datetime.now().strftime('%Y%m%d')}.parquet"))
json.dump({"model":"TFT_v3","val_da":round(da*100,2),"test_da":round(tda*100,2),
  "val_auc":round(float(auc),4),"test_auc":round(float(tauc),4),"features":CLEANED_FEATURES,
  "config":{"hidden":HIDDEN_SIZE,"heads":ATTENTION_HEADS,"lstm":NUM_LSTM_LAYERS,
            "encoder_len":MAX_ENCODER_LENGTH,"dropout":DROPOUT,"label_smoothing":LABEL_SMOOTHING}},
  open(str(MODEL_SAVE_PATH/"metrics.json"),"w"),indent=2)
print(f"저장: {MODEL_SAVE_PATH}")

## TFT v3 vs v2 변경사항\n\n| 파라미터 | v2 | v3 |\n|---------|----|----|\n| encoder_length | 30 | **60** |\n| hidden_size | 128 | **256** |\n| attention_heads | 4 | **8** |\n| lstm_layers | 1 | **2** |\n| batch_size | 256 | **512** |\n| dropout | 0.2 | **0.15** |\n| label_smoothing | - | **0.05** |\n| weight_decay | 1e-4 | **1e-3** |\n| lr_scheduler | ReduceLROnPlateau | **CosineAnnealingWarmRestarts** |\n| max_epochs | 50 | **80** |\n| patience | 8 | **10** |\n| causal_mask | No | **Yes** |\n\n| 모델 | 검증 DA | 테스트 DA | 비고 |\n|------|---------|-----------|------|\n| TFT v2 | 55.1% | 52.1% | 기존 1위 |\n| **TFT v3** | **?%** | **?%** | 스케일업 |